In [0]:
%run ./00_config_and_helpers


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

dbutils.widgets.text("ingest_date", "*")                    # * = all dates, or e.g. 2026-10-09
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"])
INGEST_DATE = dbutils.widgets.get("ingest_date")
FORCE = dbutils.widgets.get("force_reload") == "true"

# ---------- Explicit schemas (no inferSchema anywhere) ----------
user_s = StructType([
  # github level    StructField("login", StringType()),
    StructField("id", LongType()),
    StructField("type", StringType()),
    StructField("site_admin", BooleanType()),
])
person_s = StructType([
  # git level    StructField("name", StringType()),
    StructField("email", StringType()),
    StructField("date", StringType()),          # kept as string in Bronze, cast in Silver
])
commits_schema = StructType([
    StructField("sha", StringType()),
    StructField("html_url", StringType()),
    StructField("commit", StructType([
        StructField("author", person_s),
        StructField("committer", person_s),
        StructField("message", StringType()),
        StructField("comment_count", LongType()),
        StructField("verification", StructType([
            StructField("verified", BooleanType()),
            StructField("reason", StringType()),
        ])),
    ])),
    StructField("author", user_s),
    StructField("committer", user_s),
    StructField("parents", ArrayType(StructType([StructField("sha", StringType())]))),
    StructField("_corrupt_record", StringType()),
])
issues_schema = StructType([
    StructField("id", LongType()),
    StructField("number", LongType()),
    StructField("state", StringType()),
    StructField("state_reason", StringType()),
    StructField("title", StringType()),
    StructField("body", StringType()),
    StructField("html_url", StringType()),
    StructField("user", user_s),
    StructField("labels", ArrayType(StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("color", StringType()),
        StructField("description", StringType()),
    ]))),
    StructField("assignee", user_s),
    StructField("assignees", ArrayType(user_s)),
    StructField("locked", BooleanType()),
    StructField("comments", LongType()),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("author_association", StringType()),
    StructField("draft", BooleanType()),
    StructField("pull_request", StructType([
        StructField("url", StringType()),
        StructField("merged_at", StringType()),
    ])),
    StructField("_corrupt_record", StringType()),
])

# ---------- Raw -> Bronze ----------
def raw_to_bronze(endpoint, schema, table):
    start = datetime.now()
    path = f"{BASE_PATH}/landing/{LOAD_TYPE}/{endpoint}/*/{INGEST_DATE}/*.json"
    try:
        raw = spark.read.format("text").load(path).select(
            "value", F.col("_metadata.file_path").alias("_source_file"))

        parsed = raw.withColumn("r", F.from_json(
            "value", schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"}))

        rx = r"/landing/[^/]+/[^/]+/([^/]+)/([^/]+)/"
        meta = [
            F.current_timestamp().alias("load_timestamp"),
            F.regexp_extract("_source_file", rx, 1).alias("_source_repo"),
            F.regexp_extract("_source_file", rx, 2).alias("_ingest_date"),
            F.lit(LOAD_TYPE).alias("_load_type"),
            F.lit(BATCH_ID).alias("_batch_id"),
        ]

        good = (parsed.where(F.col("r._corrupt_record").isNull())
                .select("r.*", F.col("value").alias("_raw_json"), "_source_file", *meta)
                .drop("_corrupt_record"))

        bad = (parsed.where(F.col("r._corrupt_record").isNotNull())
               .select(F.lit(endpoint).alias("endpoint"),
                       F.col("value").alias("raw_record"),
                       F.lit("parse or type mismatch").alias("reason"),
                       "_source_file",
                       F.current_timestamp().alias("load_timestamp"),
                       F.lit(BATCH_ID).alias("_batch_id")))

        # idempotent at file level: skip files already loaded (unless force_reload)
        if spark.catalog.tableExists(table):
            if FORCE:
                files = [x[0] for x in good.select("_source_file").distinct().collect()]
                DeltaTable.forName(spark, table).delete(F.col("_source_file").isin(files))
            else:
                done = spark.table(table).select("_source_file").distinct()
                good = good.join(done, "_source_file", "left_anti")

        (good.write.format("delta").mode("append")
             .option("mergeSchema", "true")
             .partitionBy("_source_repo", "_ingest_date")
             .saveAsTable(table))
        bad.write.format("delta").mode("append").option("mergeSchema", "true") \
           .saveAsTable("bronze_quarantine")

        n = spark.table(table).where(F.col("_batch_id") == BATCH_ID).count()
        nq = spark.table("bronze_quarantine").where(
            (F.col("_batch_id") == BATCH_ID) & (F.col("endpoint") == endpoint)).count()
        log_run("Raw-to-Bronze", f"{path} (quarantined={nq})", start, "Success", inserted=n)
        print(f"{table}: {n} rows inserted, {nq} quarantined")
    except Exception as e:
        log_run("Raw-to-Bronze", path, start, "Failure", error=str(e)[:500])
        print(f"FAILED {table}: {e}")

raw_to_bronze("commits", commits_schema, "bronze_commits")
raw_to_bronze("issues", issues_schema, "bronze_issues")

In [0]:
display(spark.sql("SELECT COUNT(*) AS total_rows FROM bronze_commits"))
display(spark.sql("SELECT layer, status, rows_inserted, start_time FROM pipeline_execution_logs WHERE layer='Raw-to-Bronze' ORDER BY start_time DESC LIMIT 4"))

In [0]:
display(spark.sql("""SELECT layer, parameter_or_file, status, rows_inserted, rows_updated, start_time
FROM workspace.rag.pipeline_execution_logs WHERE layer='Bronze-to-Silver' ORDER BY start_time DESC LIMIT 8"""))
display(spark.sql("SELECT COUNT(*) AS silver_commits FROM workspace.rag.silver_commits"))
display(spark.sql("SELECT COUNT(*) AS silver_issues FROM workspace.rag.silver_issues"))